<div align="center">

# Split by Record

</div>

# 1. Obtain txt files with all records and seizure records only

> Obtain a file with all my records and records with seizures

In [2]:
import os
import glob

data_dir = "../Dataset_preprocessed"

h5_files = glob.glob(os.path.join(data_dir, "*.h5"))

# convert to just filenames
h5_names = set(os.path.basename(f) for f in h5_files)

print("Total H5 files:", len(h5_names))

Total H5 files: 683


> Get the exiting records with seizures txt

In [3]:
with open("RECORDS-WITH-SEIZURES.txt", "r") as f:
    edf_list = [line.strip() for line in f if line.strip()]

In [4]:
edf_list[:5]

['chb01/chb01_03.edf',
 'chb01/chb01_04.edf',
 'chb01/chb01_15.edf',
 'chb01/chb01_16.edf',
 'chb01/chb01_18.edf']

> Convert EDF -> H5 name

In [5]:
def edf_to_h5(path):
    base = os.path.basename(path)     
    return base.replace(".edf", ".h5")

In [6]:
converted = [edf_to_h5(x) for x in edf_list]

In [7]:
converted[:5]

['chb01_03.h5', 'chb01_04.h5', 'chb01_15.h5', 'chb01_16.h5', 'chb01_18.h5']

> Keep only files that actually exist in my dataset

In [8]:
valid_seizure_h5 = [f for f in converted if f in h5_names]

In [9]:
valid_seizure_h5[:5]

['chb01_03.h5', 'chb01_04.h5', 'chb01_15.h5', 'chb01_16.h5', 'chb01_18.h5']

> Save to TXT File

In [10]:
with open("RECORDS-WITH-SEIZURES-H5.txt", "w") as f:
    for name in sorted(valid_seizure_h5):
        f.write(name + "\n")

> Save all records

In [11]:
with open("RECORDS-H5.txt", "w") as f:
    for name in sorted(h5_names):
        f.write(name + "\n")

# 2. Split Data in Train/Val/Test

> Load the txt files

In [52]:
import numpy as np
import pandas as pd

with open("RECORDS-H5.txt", "r") as f:
    all_files = [line.strip() for line in f if line.strip()]

In [53]:
with open("RECORDS-WITH-SEIZURES-H5.txt", "r") as f:
    seizure_files = set(line.strip() for line in f if line.strip())

In [ ]:
# Create column with filen name
df = pd.DataFrame({"filename": all_files})

# Create column with patient number
df["patient"] = df["filename"].str.extract(r"(chb\d+)")

# Create seizure label
df["has_seizure"] = df["filename"].isin(seizure_files).astype(int)

In [63]:
df.head()

,filename,patient,has_seizure
0,chb01_01.h5,chb01,0
1,chb01_02.h5,chb01,0
2,chb01_03.h5,chb01,1
3,chb01_04.h5,chb01,1
4,chb01_05.h5,chb01,0


In [64]:
df['patient'].unique()

array(['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07',
       'chb08', 'chb09', 'chb10', 'chb11', 'chb12', 'chb13', 'chb14',
       'chb15', 'chb16', 'chb17', 'chb18', 'chb19', 'chb20', 'chb21',
       'chb22', 'chb23', 'chb24'], dtype=object)

## 1.1 Split at Record Level

In [34]:
from sklearn.model_selection import train_test_split

> Split train vs. temp

In [67]:
from sklearn.model_selection import train_test_split
import pandas as pd

split_dfs = []

for patient, p_df in df.groupby("patient"):

    print(f"\nProcessing {patient}")

    # Count seizure recordings
    seizure_counts = p_df["has_seizure"].value_counts()

    print(seizure_counts)

    # If too few seizure files, avoid stratification
    if seizure_counts.min() < 2 or len(p_df) < 5:

        print("Too few samples -> using simple train/test split")

        train_df, test_df = train_test_split(
            p_df,
            test_size=0.3,
            random_state=42
        )

        train_df["split"] = "train"
        test_df["split"] = "test"

        split_dfs.extend([train_df, test_df])

    else:

        # First split
        train_df, temp_df = train_test_split(
            p_df,
            test_size=0.3,
            random_state=42,
            stratify=p_df["has_seizure"]
        )

        # Check if second stratified split is possible
        temp_counts = temp_df["has_seizure"].value_counts()

        if temp_counts.min() < 2:

            print("Temp split too small -> using val only")

            train_df["split"] = "train"

            temp_df["split"] = "val"

            split_dfs.extend([train_df, temp_df])

        else:

            # Normal val/test split
            val_df, test_df = train_test_split(
                temp_df,
                test_size=0.5,
                random_state=42,
                stratify=temp_df["has_seizure"]
            )

            train_df["split"] = "train"
            val_df["split"] = "val"
            test_df["split"] = "test"

            split_dfs.extend([train_df, val_df, test_df])

# Combine all patients
final_df = pd.concat(split_dfs)

# Save
final_df.to_csv("CHBMIT_patient_splits.csv", index=False)

print("\nSaved successfully.")


Processing chb01
has_seizure
0    35
1     7
Name: count, dtype: int64

Processing chb02
has_seizure
0    33
1     3
Name: count, dtype: int64
Temp split too small -> using val only

Processing chb03
has_seizure
0    31
1     7
Name: count, dtype: int64

Processing chb04
has_seizure
0    39
1     3
Name: count, dtype: int64
Temp split too small -> using val only

Processing chb05
has_seizure
0    34
1     5
Name: count, dtype: int64

Processing chb06
has_seizure
0    11
1     7
Name: count, dtype: int64

Processing chb07
has_seizure
0    16
1     3
Name: count, dtype: int64
Temp split too small -> using val only

Processing chb08
has_seizure
0    15
1     5
Name: count, dtype: int64
Temp split too small -> using val only

Processing chb09
has_seizure
0    16
1     3
Name: count, dtype: int64
Temp split too small -> using val only

Processing chb10
has_seizure
0    18
1     7
Name: count, dtype: int64

Processing chb11
has_seizure
0    32
1     3
Name: count, dtype: int64
Temp split to

In [74]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

np.random.seed(42)

split_dfs = []

for patient, p_df in df.groupby("patient"):

    print(f"\nProcessing {patient}")

    seizure_df = p_df[p_df["has_seizure"] == 1]
    non_df = p_df[p_df["has_seizure"] == 0]

    # --- Seizure split ---
    n_seizures = len(seizure_df)

    n_test_seizures = max(1, int(round(0.3 * n_seizures)))

    seizure_test = seizure_df.sample(
        n=n_test_seizures,
        random_state=42
    )

    seizure_train = seizure_df.drop(seizure_test.index)

    # --- Non-seizure split ---
    non_train, non_test = train_test_split(
        non_df,
        test_size=0.3,
        random_state=42
    )

    # --- Combine ---
    train_df = pd.concat([seizure_train, non_train])
    test_df = pd.concat([seizure_test, non_test])

    train_df["split"] = "train"
    test_df["split"] = "test"

    split_dfs.extend([train_df, test_df])



Processing chb01

Processing chb02

Processing chb03

Processing chb04

Processing chb05

Processing chb06

Processing chb07

Processing chb08

Processing chb09

Processing chb10

Processing chb11

Processing chb12

Processing chb13

Processing chb14

Processing chb15

Processing chb16

Processing chb17

Processing chb18

Processing chb19

Processing chb20

Processing chb21

Processing chb22

Processing chb23

Processing chb24


In [75]:
final_df = pd.concat(split_dfs)

final_df.to_csv("dataset_split.csv", index=False)

In [73]:
final_df[final_df['patient']=='chb01']

,filename,patient,has_seizure,split
14,chb01_15.h5,chb01,1,train
15,chb01_16.h5,chb01,1,train
17,chb01_18.h5,chb01,1,train
20,chb01_21.h5,chb01,1,train
25,chb01_26.h5,chb01,1,train
40,chb01_43.h5,chb01,0,train
0,chb01_01.h5,chb01,0,train
6,chb01_07.h5,chb01,0,train
23,chb01_24.h5,chb01,0,train
34,chb01_37.h5,chb01,0,train
